## IT 720: NLP, Assignment 3 - Predicting Chess Game Outcomes

When you submit the assignment, make sure that you:

- add your name to the file name
- shut down the kernel one last time, restart it, and run your code from start to finish.
- leave the output in each cell to allow the grader to see it
- if there is a bug that you cannot resolve, leave the error message in the output cell so the grader can see it

#### Assignment Overview

I have talked in class about using NLP for non-NLP data, e.g. chess games. That is what you'll do for this assignment.  Just think of a chess game as a sentence, and the sequence of player moves as the words in the sentence. The raw data, which I am giving you, is from Mark Crowther who runs https://theweekinchess.com/ Anyone can manually download the data for free at that site, but you can also buy it to save yourself a lot of time. The raw format is PGN (Portable Game Notation) which looks like this:
```
[Event "Live Chess"]
[Site "Chess.com"]
[Date "2020.03.25"]
[Round "-"]
[White "pdrpnht"]
[Black "ColinStapczynski"]
[Result "0-1"]
[WhiteElo "1543"]
[BlackElo "2241"]
[TimeControl "180"]
[Termination "ColinStapczynski won by resignation"]
1. e4 d5 2. exd5 Qxd5 3. Nc3 Qa5 4. d4 c6 5. Nf3 Bf5 6. Bd3 Bxd3 7. Qxd3 e6 8.
O-O Nf6 9. Bg5 Nbd7 10. Ne5 Qc7 11. Ne2 Nxe5 12. dxe5 Oxe5 13. Bxf6 gxf6 14.
Rfe1 Bd6 15. Ng3 Qd5 16. Rad1 Qxd3 17. Rxd3 O-O-O 18. Red1 Be7 19. Ne4 Rxd3 20.
Rxd3 Rd8 21. Rh3 f5 0-1
```

I have parsed such data from over 3 million games that I purchased from Crowther to find almost 600k games where each player has an Elo rating of at least 2400, which will be the best players in the world, including a great many Grandmasters.  When you read the data into a pandas dataframe you'll see information similar to what you see above.  Most importantly are the numbered moves at the end.  The final symbol, '0-1' indicates that Black was the winner of this game.  A White win is notated as '1-0' and a Draw as '1/2-1/2'.  Those three possible outcomes are your target variable values to predict from the numbered moves of the game.  The pandas dataframe will show those outcomes in the 'Result' column.  The 'Moves' column will contain the numbered moves.  I have already done some data cleaning for you (e.g. removing games that were mistranscribed resulting in illegal moves, and much much more), but you will still have to do a small amount of cleaning by removing the numbers, leaving only the move notations.

You do not need to understand the move notations to do this assignment. Just know that they are referred to as 'SAN' (Standard Algebraic Notation).  Pretend they are simply words in a foreign language. The objective of this assignment is just to give you some experience building 1-dimensional Convolutional Neural Networks, which were originally created to recognize images, but have been adapted by the NLP community to process linguistic data.

Thus, you will be using a Computer Vision technique adapted to NLP to predict the outcomes of chess games!  In a beautiful example of reciprocal karma, the Computer Vision community has adapted the Transformer architecture from the NLP community to improve image recognition.  Such is the wonderful community of maching learning!

You will find additional, more specific task details below to help you.  Have fun with the assignment.

### Rubric for the numbered sections below where you must write your own code, 175 points total
1.   5 points: Read data
2.   5 points: Player ratings
3.  15 points: Class balance graph
4.  10 points: Clean data
5.  10 points: Explore data
6.  10 points: Add column
7.  20 points: Split data
8.  20 points: Tokenize moves
9.  20 points: Build CNN baseline
10. 10 points: Train baseline
11. 10 points: Evaluate baseline
12. 40 points: Build/Evaluate more complex CNN model
13. Optional : PCA visualization of game embeddings.


In [ ]:
!pip install python-snappy

In [ ]:
!pip install fastparquet

In [ ]:
# If you write your CNN code in keras/tensorflow, then this cell has all of the packages you will need
# You can write a solution using other packages, including PyTorch, if you prefer.
# But you MUST read the data file into pandas using the pandas expression
# "pd.read_parquet(pathToFileName)" that requires the fastparquet and snappy packages.

import fastparquet # The data file has exension: .parquet.snappy
import re
import snappy      # snappy is the compression format used when the data file was saved
import umap

import matplotlib.pyplot as plt
import numpy             as np
import pandas            as pd
import seaborn           as sns
import tensorflow        as tf

from sklearn.decomposition   import PCA
from sklearn.metrics         import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import LabelEncoder

from tensorflow.keras.callbacks              import EarlyStopping
from tensorflow.keras.layers                 import Activation, Conv1D, Dense, Dropout, Embedding, GlobalAveragePooling1D
from tensorflow.keras.layers                 import GlobalMaxPooling1D, Input, BatchNormalization
from tensorflow.keras.models                 import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text     import Tokenizer

%matplotlib inline

print("TensorFlow version:", tf.__version__)

#### Task 1: 5 points. Read the raw data in to a dataframe
Then show the first several rows of the dataframe.
Print the shape of the dataframe and .display its head.
Finally, show the number of games in each of the three results (Black, White, Draw)

In [ ]:
from google.colab import drive           # imports the drive module from Google Colab
drive.mount('/content/drive')            # mounts Google Drive to the Colab environment

In [ ]:
!ls /content/drive/MyDrive/IT720/Assignment_3/Datasets   #This command lists all files inside the Datasets folder stored in Google Drive.
                                                        # It helps us check whether our dataset files are available before using them.

# ### How to Use Your Own Google Drive Path in Colab

# The path shown in this notebook is only an example.
#You must replace it with the path where your dataset is stored in **your Google Drive**.
#Example:
#If your dataset is stored here:
#MyDrive → Course → Assignment3 → data

#Then your path will be:
 # /content/drive/MyDrive/Course/Assignment3/data

In [ ]:
# Read data

# Set the path to your parquet.snappy file here. If you are running this notebook
# in Google Colab you can mount your Drive and point to the file in Drive.
pathToFileName = "games_2400_plus.parquet.snappy"  # <-- replace with your path

try:
    df = pd.read_parquet(pathToFileName)
except Exception as e:
    print("Could not read file at", pathToFileName)
    print("Error:", e)
    print('\nPlease set `pathToFileName` to the actual parquet.snappy file path on your system.')
    raise

print('Shape:', df.shape)
display(df.head())
print('\nResult value counts:')
print(df['Result'].value_counts())


#### Task 2: 5 points.  Smallest and largest player Elo ratings.

Print the smallest and the largest Elo rating found in the data

In [ ]:

# Player strengths:

# Compute smallest and largest Elo among both players
elo_cols = []
for c in ['WhiteElo', 'BlackElo']:
    if c in df.columns:
        elo_cols.append(c)

if elo_cols:
    all_elos = pd.concat([df[c].dropna() for c in elo_cols])
    print('Min Elo:', int(all_elos.min()))
    print('Max Elo:', int(all_elos.max()))
else:
    print('No Elo columns found in dataframe')


#### Task 3: 15 points. Write a function to display a class balance bar graph  

In other words one bar will show black wins, a second white wins, and a third the draws.
If all three categories of game results show the same proportion or count of games, then the 3 classes are perfectly balanced.
If not, then you have unbalanced data, and that may influence whether or not you want to use stratfied sampling when you split your data into training, validation and test subsets.

Each bar should show either the number of games with that result, or the overall % games with that result.
Make sure to have a graph title at the top, and show the game count on the y-axis
with the Result notation under each bar, i.e. '0-1', for the bar with black wins,
'1-0' for white wins and '1/2-1/2' for draws.

Follow this function definition with a call to it, showing the bar graph for the dataset.

In [ ]:

# Function for bar graph
def plot_class_balance(df, percent=False, title='Class balance'):
    counts = df['Result'].value_counts()
    order = ['0-1', '1-0', '1/2-1/2']
    counts = counts.reindex(order).fillna(0)
    plt.figure(figsize=(6,4))
    if percent:
        vals = counts / counts.sum() * 100
        sns.barplot(x=vals.index, y=vals.values)
        plt.ylabel('Percent of games')
    else:
        sns.barplot(x=counts.index, y=counts.values)
        plt.ylabel('Number of games')
    plt.title(title)
    for i, v in enumerate(counts.values):
        plt.text(i, v + counts.values.max()*0.01, f'{v:,}', ha='center')
    plt.xlabel('Result')
    plt.tight_layout()

# call the function
plot_class_balance(df, percent=False, title='Game result class balance')




#### Task 4: 10 points. Clean the Moves
Write code to replace the values in the 'Moves' column by deleting the move numbers such as '1.', leaving only the SAN notations.

For example, instead of: '1. e4 e5' for the first move by each player, it should show only: 'e4 e5' (without the quotation marks)
Print a few rows to see that you did it correctly.

In [ ]:

# Clean up the Moves column by removing the numbers,
# leaving only the SAN (Standard Algebraic Notation)
# White moves are in odd-numbered locations, and Black moves are even-numbered

def clean_moves(s):
    if pd.isna(s):
        return s
    # remove move numbers like '1.' or '23.' and extra whitespace
    s = re.sub(r"\d+\.(?:\.\.)?", "", s)
    # remove result tokens if they appear at the end
    s = re.sub(r"\s*(1-0|0-1|1/2-1/2)\s*$", "", s)
    # collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

df['Moves'] = df['Moves'].astype(str).map(clean_moves)
display(df[['Moves']].head(10))

print('DataFrame info:')
print(df.info())
print('\nDescribe:')
display(df.describe(include='all'))

#### Task 5: 10 points. Expore data characteristics.

Print the value of the dataframe's .info and .describe properties

In [ ]:

# Create Result_ID using LabelEncoder
le = LabelEncoder()
df['Result_ID'] = le.fit_transform(df['Result'])
print('Mapping:', dict(zip(le.classes_, le.transform(le.classes_))))
display(df[['Result','Result_ID']].head())




#### Task 6: 10 points.  Add a numeric column for the game results.

Add a new column called 'Result_ID' that has the integer value 0 for black wins, 1 for white wins, and 2 for Draws.
You can do this with scikit-learn's LabelEncoder.
Print a few lines to verify the new column.

In [ ]:

# Keep only Moves and Result_ID for modeling
X = df['Moves'].astype(str).values
y = df['Result_ID'].values

# Decide whether to stratify: check imbalance
counts = df['Result'].value_counts(normalize=True)
print('Result proportions:\n', counts)
use_stratify = True

# We'll split into train/val/test = 70/15/15
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y if use_stratify else None)
relative_val = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=relative_val, random_state=42, stratify=y_temp if use_stratify else None)

print('Shapes:')
print('X_train:', X_train.shape)
print('X_val:', X_val.shape)
print('X_test:', X_test.shape)

print('\nExample training row:')
print(X_train[0])
print('Result_ID:', y_train[0])

#### Task 7: 20 points. Split the data into training, validation and test sets with sizes of your own choosing.
You only need to retain the "Moves" column for the predictors (e.g. X_), and the "Result_ID" column for the targets (e.g. y_).
Print the shapes of the X_ variables for all 3 splits, and then show one of the training data rows with the raw chess moves, as well as that game's result ID value.

It is generally good practice to shuffle your data as you create these splits in order to randomize the data in case it was collected in some non-random order.
Depending on your results from Task 3 above, either use stratified sampling (for unbalanced data) or do not use stratified sampling (for balanced data).

#### Task 8: 20 points. Tokenize the SAN 'words'
Tokenize the chess SAN move notations in all 3 data splits to convert the chess move notation strings to integers, with a unique integer for every unique SAN string.  
You can use the keras Tokenizer to do it. You may need to search the web for examples of how to do this,
but it would be the same as tokenizing the vocabulary words for a natural language. Do NOT try to do any subword tokenization.
You can use '<OOV>' as the token for any ouf of vocabulary words that may be encountered.
I have already set maxVocabSize and maxGameLength for you.  You will also need to pad any games
that are shorter than maxGameLength with zeros using keras pad_sequences.
After tokenizing, print the number of unique tokens that were found in the training dataset
and print the shape of the newly padded training dataset.

In [ ]:


# Hyperparameters for tokenization
maxVocabSize  = 10500 # This is the number of unique move notations in the larger game dataset
                      # that was used to extract the grandmaster games for this exercise
maxGameLength = 160   # This is about double the mean game length of roughly 80

# Tokenize
tokenizer = Tokenizer(num_words=maxVocabSize, oov_token='<OOV>', filters='')
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=maxGameLength, padding='post', truncating='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=maxGameLength, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=maxGameLength, padding='post', truncating='post')

vocab_found = min(maxVocabSize, len(tokenizer.word_index) + 1)
print('Unique tokens in tokenizer (including OOV):', len(tokenizer.word_index))
print('Vocab used (capped by maxVocabSize):', vocab_found)
print('Padded training shape:', X_train_pad.shape)



#### Task 9: 20 points.

Build a convolutional neural network with a single 1-dimensional CNN layer, followed by a global max pooling layer OR an average max pooling layer. Then add a dense non-linear layer which is then followed by a softmax layer to classify the chess games into the 3 possible result outcomes.

Make sure you also add an Embedding layer at the beginning so that the model will learn high dimensional embeddings for each of the move tokens.
You can decide how many dimensions to use for the embeddings, the number of filters you wish to use, what the kernal (window) size should be for the convolutions, as well as which optimizer to use, whether you want to use dropout or not, and any other settings. Print a visual summary of your CNN architecture, and compile it using the sparse categorical cross entropy loss function.

This model will be your baseline.

In [ ]:

# CNN baseline model
embedding_dim = 128
model = Sequential([
    Input(shape=(maxGameLength,)),
    Embedding(input_dim=maxVocabSize, output_dim=embedding_dim, input_length=maxGameLength),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# keep a reference to the embedding layer for optional visualization later
embedding_layer = model.layers[1]




#### Task 10: 10 points. Train the model with early stopping.

Train your model for at least 20 epochs, using early stopping based on validation data loss to avoid overfitting (and also saving you unnecessary waiting time).  You can decide other things such as the batch size, etc.

In [ ]:

es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=30,
    batch_size=128,
    callbacks=[es]
)


#### Task 11: 10 points. Model Evaluation

After training your model, evaluate it by predicting the test data results and print the test accuracy. You may print the test data loss if you wish.
Following this, compute and display a scikit-learn classification_report and confusion_matrix.
You may, if you wish, show a matplotlib (or Seaborn) graphic confusion matrix, but since there are only 3 categories, the scikit learn's confusion_matrix will suffice.


In [ ]:

test_loss, test_acc = model.evaluate(X_test_pad, y_test, verbose=2)
print(f'Test loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}')

y_pred_probs = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_probs, axis=1)

print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

print('\nConfusion matrix:')
print(confusion_matrix(y_test, y_pred))




#### Task 12: 40 points. Better CNN architecture.

Create a new, more sophisticated CNN architecture that obtains a higher accuracy on the test data than your baseline.
Some things you may want to try (suggestions only, you can decide what you want to do) include:
1. More than one CNN layer
2. Batch or layer normalization
3. Dropout
4. Different number of filters
5. Different kernel sizes
6. Deeper classification layer at the end.
7. Use of a tuning package to help you discover better hyperparameter settings than your baseline model.

After training your model, evaluate it and print the accuracy results as well as the classification report and confusion matrix, just like you did for your baseline model. You should be able to get a test set accuracy of 80% or more, though your grade does not depend on it.

In [ ]:

# Better CNN architecture
from tensorflow.keras.layers import InputLayer

better_model = Sequential([
    InputLayer(input_shape=(maxGameLength,)),
    Embedding(input_dim=maxVocabSize, output_dim=embedding_dim, input_length=maxGameLength),
    Conv1D(128, 5, activation='relu', padding='same'),
    BatchNormalization(),
    Conv1D(128, 3, activation='relu', padding='same'),
    BatchNormalization(),
    GlobalAveragePooling1D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

better_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
better_model.summary()

es2 = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
history2 = better_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=40,
    batch_size=128,
    callbacks=[es2]
)

loss2, acc2 = better_model.evaluate(X_test_pad, y_test, verbose=2)
print(f'Better model - test loss: {loss2:.4f}, test acc: {acc2:.4f}')

y2_pred = np.argmax(better_model.predict(X_test_pad), axis=1)
print('\nClassification report (better model):')
print(classification_report(y_test, y2_pred, target_names=le.classes_))
print('\nConfusion matrix (better model):')
print(confusion_matrix(y_test, y2_pred))




#### Optional Task 13: Only for learning purpose

Before Transformers, people sometimes created sentence embeddings by averaging a sentence's word embeddings. Let's do the same thing here, and then visualize the games.

Create training set game embeddings by averaging the SAN 'word' embeddings that were learned during training.
Use the PCA dimensionality reduction algorithm to visualize in 2 dimensions the training set game embeddings built from their move embeddings.

In [ ]:

# PCA visualization for Game Vectors averaged from the game's move embeddings
from tensorflow.keras.models import Model

# build a small model that maps input sequence to averaged embedding vector
inp = better_model.input
emb_out = better_model.layers[1].output  # embedding layer output (sequence of embeddings)
avg_pool = GlobalAveragePooling1D()(emb_out)
embed_model = Model(inputs=inp, outputs=avg_pool)

train_embeds = embed_model.predict(X_train_pad, batch_size=256)

pca = PCA(n_components=2)
proj = pca.fit_transform(train_embeds)

plt.figure(figsize=(8,6))
palette = {0:'red',1:'blue',2:'green'}
for cls in np.unique(y_train):
    mask = y_train == cls
    plt.scatter(proj[mask,0], proj[mask,1], s=8, c=palette[cls], label=le.inverse_transform([cls])[0])
plt.legend()
plt.title('PCA of averaged game embeddings (training set)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
